[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/building-rag-pipelines/blob/main/notebooks/06_evaluation.ipynb)

# Building RAG Pipelines
## Notebook 06: Evaluation — Measuring & Diagnosing RAG
**Duration:** 35 min &nbsp;|&nbsp; **Mode:** Conceptual + Guided Analysis

> Taught **WHY → WHAT → HOW**. We keep asking *"What happens if this step is poorly
> designed?"* and we **predict before we run** and **compare outputs**. LangChain is
> shown as a **parallel mapping** — it abstracts mechanics but not design decisions.

![pipeline](https://dummyimage.com/1000x70/1f2937/ffffff&text=Loading+%E2%86%92+Chunking+%E2%86%92+Retrieval+%E2%86%92+Augmentation+%E2%86%92+Generation+%E2%86%92+Evaluation)

In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first. (Same as every notebook.)
# ============================================================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/building-rag-pipelines.git"  # INSTRUCTOR: set this

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

_pip("numpy", "openai", "tiktoken", "rank-bm25", "beautifulsoup4", "pypdf",
     "langchain-community", "langchain-text-splitters", "langchain-openai", "faiss-cpu")
try:
    import rag_pipeline
except ModuleNotFoundError:
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
        if os.path.isdir("building-rag-pipelines"):
            sys.path.insert(0, "building-rag-pipelines")
        else:
            print("Clone failed. Upload `rag_pipeline/` + `data/` via the Colab file browser, then re-run.")
    else:
        sys.path.insert(0, os.path.abspath(".."))
    import rag_pipeline

def data_path(*parts):
    for base in ("data", "../data", "building-rag-pipelines/data"):
        p = os.path.join(base, *parts)
        if os.path.exists(p):
            return p
    return os.path.join("data", *parts)

print("rag_pipeline", rag_pipeline.__version__, "ready.  Colab:", IN_COLAB)

In [ ]:
# ============================================================
# CHOOSE YOUR PROVIDERS  (OpenAI is the default)
# ============================================================
# Default stack = OpenAI: gpt-4o-mini (LLM) + text-embedding-3-small (embeddings).
# In Colab the key is read automatically from the Colab SECRETS manager:
#   left sidebar -> key icon -> add a secret named OPENAI_API_KEY
#   -> toggle "Notebook access" ON  -> re-run this cell.
# If no key is found anywhere, we fall back to the offline MOCK so the notebook
# still runs end-to-end.
import os

def _load_openai_key():
    if os.getenv("OPENAI_API_KEY"):
        return True
    try:  # Colab Secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return True
    except Exception:
        pass  # not in Colab, secret missing, or access not granted
    return False

if _load_openai_key():
    os.environ.setdefault("RAG_LLM_PROVIDER", "openai")
    os.environ.setdefault("RAG_EMBED_PROVIDER", "openai")
    print("OpenAI key found -> using the OpenAI stack.")
else:
    os.environ["RAG_LLM_PROVIDER"] = "mock"
    os.environ["RAG_EMBED_PROVIDER"] = "mock"
    print("No OPENAI_API_KEY found -> using the offline MOCK providers.\n"
          "In Colab: add a Secret named OPENAI_API_KEY (key icon, left sidebar),\n"
          "enable Notebook access, and re-run this cell to switch to OpenAI.")

from rag_pipeline import config
print(config.current_config())

In [ ]:
from rag_pipeline.loaders import load_directory
docs = load_directory(data_path("corpus"))
print(f"Loaded {len(docs)} documents from the Acme Cloud corpus.")

## WHY — without measurement, every decision is opinion

"How do you know your RAG system is any good?" Evaluation turns RAG from art into
engineering, and — critically — it **localises failure**. RAG can break in the
**retrieval** stage (wrong chunks) or the **generation** stage (right chunks, wrong
answer), and the fix is *completely different*. Metrics that separate the two are
what let you debug instead of flail.

> **What happens if evaluation is poorly designed?** You optimise the wrong thing —
> tune the prompt for a week when the real problem was recall, or ship a system that
> scores well on fluency while quietly hallucinating.

## WHAT — two metric families + failure modes

**Failure modes to catch:** retrieval errors, hallucination, context noise, plus
latency & cost.

**RETRIEVAL metrics** (is the right context found?) — need ground-truth relevant docs:
- **precision@k** — of the top-k retrieved, what fraction are relevant.
- **recall@k** — of all relevant docs, what fraction appear in top-k.
- **hit_rate@k** — did at least one relevant doc make top-k.
- **MRR** — reciprocal rank of the first relevant hit (rewards ranking it high).

**GENERATION metrics** (is the answer faithful & correct?):
- **faithfulness** — is every claim supported by the retrieved context (anti-hallucination).
- **answer relevance** — does the answer address the question.
- **RAGAS** framework — faithfulness, answer correctness, context precision/recall.

**Heuristics vs frameworks:** cheap embedding heuristics for a fast signal; RAGAS /
LLM-as-judge for rigour. **Metrics vs rubrics:** numbers for what's countable, human
rubrics for what isn't.

## HOW — retrieval metrics over our eval dataset

The eval set (`data/eval/eval_dataset.jsonl`) has 21 questions with ground-truth
source documents, **including 3 NEGATIVE tests** (unanswerable questions where the
correct behaviour is a refusal).

In [ ]:
from rag_pipeline import config, evaluation as ev
from rag_pipeline.chunking import recursive_chunk
from rag_pipeline.vectorstore import InMemoryVectorStore
from rag_pipeline.embeddings import embed_documents
from rag_pipeline.retrieval import DenseRetriever, BM25Retriever, HybridRetriever

emb, llm = config.get_embedder(), config.get_llm()
chunks = recursive_chunk(docs, 500, 50)
store  = InMemoryVectorStore().add(chunks, embed_documents(emb, chunks))
dense  = DenseRetriever(store, emb)
hybrid = HybridRetriever(dense, BM25Retriever(chunks))

examples = ev.load_eval_dataset(data_path("eval", "eval_dataset.jsonl"))
print(f"{len(examples)} eval examples "
      f"({sum(e.is_negative for e in examples)} negatives).")

> ### ✋ Predict before you run
> We'll score dense vs hybrid retrieval on the eval set with recall@3 and MRR. Which will score higher, and roughly how big a gap do you expect? (Recall from Notebook 04 that hybrid caught both exact-term and paraphrase queries.)
>
> *Commit to a guess before executing. Comparing prediction vs result is the point.*

In [ ]:
print("DENSE :", ev.evaluate_retriever(dense,  examples, k=3))
print("HYBRID:", ev.evaluate_retriever(hybrid, examples, k=3))

**What you should observe:** hybrid beats dense on recall@3 and MRR (in our runs,
recall@3 ≈ 0.96 vs ≈ 0.85). **This single table is how you justify a design change
to a stakeholder** — not vibes, evidence.

## HOW — generation quality: faithfulness & negatives

Faithfulness asks: is the answer *supported by the retrieved context*? We show a
cheap embedding heuristic and an LLM-as-judge (needs a key). For the 3 negative
questions we check the opposite: does the system correctly **refuse**?

In [ ]:
from rag_pipeline.generation import answer_with_sources
from rag_pipeline.evaluation import faithfulness_heuristic, refusal_correct

# Answerable question -> should be faithful, not a refusal.
r1 = answer_with_sources(llm, hybrid, "How often are encryption keys rotated?", k=3)
print("Answerable -> faithfulness(heuristic):",
      round(faithfulness_heuristic(r1["answer"], r1["context"], emb), 3))

# Negative questions -> the CORRECT behaviour is a refusal.
print("\nNegative-test refusal check:")
for ex in [e for e in examples if e.is_negative]:
    r = answer_with_sources(llm, hybrid, ex.question, k=3, grounded=True)
    print(f"  refused={refusal_correct(r['answer'])}  Q: {ex.question}")

## HOW — RAGAS metrics (faithfulness, relevancy, context precision/recall)

**RAGAS** grades a RAG answer on four metrics with an LLM-as-judge:
**faithfulness** (is every claim supported by the context?), **answer relevancy**,
**context precision** (are the retrieved chunks relevant?), and **context recall**
(do the chunks cover the reference answer?).

The `ragas` *library* is powerful but its dependencies are brittle to install in a
shared/Colab session (it pins specific langchain/pydantic/numpy versions and often
breaks the environment). Under the hood, though, those metrics are just LLM-judge
prompts — so we compute the **same four metrics directly** with
`evaluate_ragas_style(...)`. It needs only the OpenAI key set above, always runs,
and demystifies exactly what RAGAS does. (The real library is shown as an optional
extra in the next cell.)

> The one thing everyone gets wrong: `contexts` must be the **retrieved chunk
> texts** (`answer_with_sources` returns them as `r["contexts"]`), not their labels.

In [ ]:
# Build the records: question, answer, the ACTUAL retrieved chunk texts, ground truth.
records = []
for ex in [e for e in examples if not e.is_negative][:5]:
    r = answer_with_sources(llm, hybrid, ex.question, k=3)
    records.append({
        "question": ex.question,
        "answer": r["answer"],
        "contexts": r["contexts"],        # real chunk texts (not labels)
        "ground_truth": ex.ground_truth,
    })

# RAGAS-style metrics — always runs with the OpenAI stack, no extra install.
scores = ev.evaluate_ragas_style(records, llm, emb)
print("RAGAS-style scores:")
for k, v in scores.items():
    print(f"  {k:<18} {v}")

### (Optional) The real RAGAS library

If you want to run the actual `ragas` package, do it in a **fresh environment**
(`pip install ragas datasets`) to avoid version clashes with LangChain. Our
`evaluate_with_ragas(records)` wrapper accepts the same `records` and returns
`None` if the library isn't importable — so this cell is safe to run either way.

In [ ]:
# Optional: real RAGAS. Returns None (with a friendly note) if ragas isn't installed.
ragas_scores = ev.evaluate_with_ragas(records)
print("Real RAGAS scores:", ragas_scores)

## HOW — building a RAG evaluation dataset (the important part)

The agenda flags this as **important**. A good eval set has:
1. **Questions** representative of real usage (factual, list, multi-hop).
2. **Ground truths** — the reference answer AND which document(s) are relevant.
3. **Negative tests** — questions the corpus *cannot* answer; correct output = refusal.
   These catch hallucination that positive tests never will.
4. **Metrics vs rubrics** — automate what's countable (precision@k); use human
   rubrics for what isn't (tone, completeness, safety).

You can *bootstrap* a draft with an LLM, then a human curates it — never ship
auto-generated ground truth unreviewed.

In [ ]:
# Auto-draft candidate questions from chunks (a STARTING point for human curation).
drafts = ev.draft_questions_from_chunks(llm, chunks[:3])
for d in drafts:
    print("Q:", d.question, "  | relevant:", d.relevant_ids)

## HOW — diagnose a "bad RAG output" step by step

The agenda: *present a bad RAG output and diagnose it.* The trick is to ask, in
order: **did retrieval fail, or did generation fail?**
- Low **recall@k** for this query → retrieval failure (fix chunking/retriever/k).
- Good retrieval but low **faithfulness** → generation failure (fix prompt/grounding).

In [ ]:
# A deliberately weak setup: tiny chunks + k=1 -> likely to miss context.
from rag_pipeline.pipeline import RAGPipeline
from rag_pipeline.chunking import fixed_size_chunk

bad = RAGPipeline(chunker=lambda d: fixed_size_chunk(d, 120, 0), k=1).ingest(docs)
out = bad.query("Which plan gives SSO and a 99.99% SLA?")
print(out["trace"].show()[:500])
print("\nDIAGNOSIS HINT: only", len(out["sources"]),
      "chunk retrieved for a multi-document question -> RETRIEVAL failure (raise k, bigger chunks, hybrid).")

## Recap
- Measure **retrieval** and **generation** separately — the fixes differ.
- precision@k / recall@k / MRR for retrieval; faithfulness / RAGAS for generation.
- **Negative tests** are how you catch hallucination.
- Diagnosis = localise the failing stage, then fix *that* stage.

**Next → Notebook 07 (Advanced RAG):** where you go once the basic pipeline is solid.